# PushT visual Diffusion Policy on ManiSkill

Choose **Runtime > Change runtime type > T4 GPU** before running. This notebook targets Python 3.12 and delegates substantive code to the repository so experiments remain reproducible.

In [ ]:
import platform
import subprocess

print("Python:", platform.python_version())
subprocess.run(["nvidia-smi"], check=True)

Set `REPO_URL` to the GitHub repository you pushed from Cursor.

In [ ]:
REPO_URL = "https://github.com/renanakashima/ripl_assignment.git"
repo_name = REPO_URL.rstrip("/").rsplit("/", 1)[-1].removesuffix(".git")
!git clone {REPO_URL} /content/{repo_name}
%cd /content/{repo_name}/t-i
!bash scripts/setup_colab.sh

## Prepare the dataset
Use the native `pd_ee_delta_pose` source selected by ManiSkill's published Push-T command. The source was collected with 1,024 environments; final data should be replayed with 1,024 environments on HCE. This 64-environment Colab command is an explicit lower-fidelity smoke path, not the reportable dataset. The script requires exactly 100 successful source and RGB trajectories and validates observation/action alignment.

In [ ]:
!ALLOW_REPLAY_ENV_MISMATCH=1 NUM_DEMOS=100 REPLAY_ENVS=64 bash scripts/prepare_pusht_delta_pose_demos.sh

## Smoke test
One update and one evaluation episode verify the complete native-controller data/model/simulator path. This is not meaningful training.

In [ ]:
!python train_dp.py --config configs/pusht_rgb_delta_pose.yaml --exp-name pusht-rgb-delta-pose-smoke --num-demos 4 --batch-size 8 --total-iters 1 --warmup-steps 1 --eval-freq 1 --save-freq 1 --num-eval-episodes 1 --num-eval-envs 1 --no-capture-video

## Persistent full run
For a final experiment, copy the HCE-generated 1,024-environment replay into the same ManiSkill demo path first. Save checkpoints to Drive so a Colab disconnect does not erase them. Batch 128 is more suitable for a T4 than the HCE config's batch 256.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
OUTPUT_DIR = "/content/drive/MyDrive/ripl-pusht-runs"

In [ ]:
!python train_dp.py --config configs/pusht_rgb_delta_pose.yaml --batch-size 128 --output-dir {OUTPUT_DIR}

After training, set `CHECKPOINT` to a `final.pt` or `best_success_once.pt` file. See README.md for resume and TensorBoard commands.

In [ ]:
CHECKPOINT = f"{OUTPUT_DIR}/RUN_NAME/checkpoints/final.pt"
assert "RUN_NAME" not in CHECKPOINT, "Set CHECKPOINT first"
!python eval_dp.py --checkpoint {CHECKPOINT} --num-eval-episodes 20 --num-eval-envs 10